# Stage 3 — Aggregate, score, plot

Reads per-`(model, benchmark)` JSONs from `results/`, writes the aggregated long-format Parquets, the bootstrap-CI table, and all figures. Re-runnable in seconds — change `plotting.py` or `statistics.py` and re-execute without touching the GPU stage.

In [ ]:
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive; drive.mount('/content/drive')
    %cd /content/llm-bias-eval
    if not os.path.lexists('results'):
        os.symlink('/content/drive/MyDrive/llm-bias-eval/results', 'results')

In [ ]:
from pathlib import Path
import pandas as pd

from biaseval.analysis.aggregate_results import (
    aggregate_logit_results, aggregate_probe_results,
    aggregate_intervention_results, write_aggregated,
)
from biaseval.analysis.plotting import generate_all
from biaseval.analysis.statistics import per_example_bootstrap
from biaseval.analysis.regression import write_regression_report

RESULTS = Path('results')
FIGURES = Path('figures')
TABLES = Path('results/tables'); TABLES.mkdir(parents=True, exist_ok=True)

In [ ]:
write_aggregated(RESULTS, RESULTS / 'aggregated.parquet')
logit_df = aggregate_logit_results(RESULTS)
probe_df = aggregate_probe_results(RESULTS)
intv_df  = aggregate_intervention_results(RESULTS)
print(f'logit={len(logit_df)}  probe={len(probe_df)}  intervention={len(intv_df)}')

In [ ]:
rows = [
    {'benchmark': bench, 'model_id': model_id, 'prompt_mode': prompt_mode, **stats}
    for bench in ('crows_pairs', 'stereoset')
    for (model_id, prompt_mode), stats in per_example_bootstrap(RESULTS, bench, n_iter=1000).items()
]
bootstrap_df = pd.DataFrame(rows)
bootstrap_df.to_csv(TABLES / 'bootstrap_ci.csv', index=False)

In [ ]:
generate_all(logit_df, probe_df, figures_dir=FIGURES, intv_df=intv_df)

## Confound-controlled regression

Per the supervisor's framing question: estimate the variant (base→instruct) effect on each headline bias metric **after partialling out** model size (log₁₀ params), family, generation, and prompt_mode. Cluster-robust SEs on `model_id` plus a per-pair logistic GEE on CrowS-Pairs. Holm-Bonferroni across the 4 benchmark fits.

Writes `results/tables/regression_report.md` (paste-ready for the thesis) and `regression_summaries.json` (raw fits for downstream slicing).

In [ ]:
md_path = write_regression_report(RESULTS, TABLES)
print(open(md_path).read())